In [1]:
"""
PIPELINE STAGE: Spatial Enrichment, Plate Mapping & Multi-Source Standardization
INPUTS: prompt_abone.xlsx, consumption_2021_2024.xlsx
OUTPUTS: abone.xlsx, consumption_2021_2024_son.xlsx
LIBRARIES: pandas, numpy

1. OBJECTIVE:
    Integrate geographic metadata by mapping Turkish province names to their official license 
    plate codes (1-81). This stage synchronizes disparate datasets (Subscriber Counts & 
    Consumption Metrics) into a unified spatial schema, ensuring consistency for 
    econometric modeling and regional analysis.

2. GEOGRAPHIC STANDARDIZATION ENGINE:
    A. Logical Imputation: Apply Forward Fill (ffill) to handle merged cells in Excel, 
       ensuring every row retains its associated 'Province' identity.
    B. Case-Sensitive Normalization: Convert all province names to UPPERCASE with strict 
       adherence to Turkish character integrity (e.g., İ -> İ, I -> I).
    C. Plate Mapping: Cross-reference cleaned names against a master dictionary to generate 
       a 'Plate_Code' column, followed by an automatic purge of non-matching or NaN entries.

3. OPERATION 1: SUBSCRIBER DATA REFINEMENT:
    - Target: processed_data/steps/subscriber_data.xlsx
    - Dynamic Column Repair: Direct renaming of specific indices to prevent data loss.
    - Category Harmonization: 
        * Consolidate "Public" variants (including "Ticarethane") into a unified 'Public' label.
        * Normalize "Tarımsal" variants into a concise 'Agricultural' (Tarım) category.
    - Filtering: Eliminate aggregate rows (e.g., "İl Toplam", "Genel Toplam") to isolate raw data.

4. OPERATION 2: CONSUMPTION DATA AGGREGATION:
    - Target: processed_data/steps/consumption_2021_2024.xlsx
    - Domain Focus: Isolate regional totals by filtering for "İl Toplam" (Province Total) entries only.
    - Column Optimization: Remove redundant source metadata and rename quantitative metrics 
      to a standard 'Consumption' header.

5. FINAL STRUCTURAL SPECIFICATIONS:
    - Sorting Logic: Mandatory triple-sort by [Plate_Code, Year, Month] in ascending order.
    - Uniform Schema: Enforce a strict column sequence for both datasets to ensure 
      seamless merge-readiness in subsequent pipeline steps.
    - Error Management: Full encapsulation in try-except blocks with real-time audit logging.
"""


import os
import pandas as pd
import numpy as np

# --- ORTAK PLAKA SÖZLÜĞÜ ---
plaka_kodlari = {
    "ADANA": 1, "ADIYAMAN": 2, "AFYONKARAHİSAR": 3, "AĞRI": 4, "AMASYA": 5, "ANKARA": 6, "ANTALYA": 7, "ARTVİN": 8,
    "AYDIN": 9, "BALIKESİR": 10, "BİLECİK": 11, "BİNGÖL": 12, "BİTLİS": 13, "BOLU": 14, "BURDUR": 15, "BURSA": 16,
    "ÇANAKKALE": 17, "ÇANKIRI": 18, "ÇORUM": 19, "DENİZLİ": 20, "DİYARBAKIR": 21, "EDİRNE": 22, "ELAZIĞ": 23, "ERZİNCAN": 24,
    "ERZURUM": 25, "ESKİŞEHİR": 26, "GAZİANTEP": 27, "GİRESUN": 28, "GÜMÜŞHANE": 29, "HAKKARİ": 30, "HATAY": 31, "ISPARTA": 32,
    "MERSİN": 33, "İSTANBUL": 34, "İZMİR": 35, "KARS": 36, "KASTAMONU": 37, "KAYSERİ": 38, "KIRKLARELİ": 39, "KIRŞEHİR": 40,
    "KOCAELİ": 41, "KONYA": 42, "KÜTAHYA": 43, "MALATYA": 44, "MANİSA": 45, "KAHRAMANMARAŞ": 46, "MARDİN": 47, "MUĞLA": 48,
    "MUŞ": 49, "NEVŞEHİR": 50, "NİĞDE": 51, "ORDU": 52, "RİZE": 53, "SAKARYA": 54, "SAMSUN": 55, "SİİRT": 56,
    "SİNOP": 57, "SİVAS": 58, "TEKİRDAĞ": 59, "TOKAT": 60, "TRABZON": 61, "TUNCELİ": 62, "ŞANLIURFA": 63, "UŞAK": 64,
    "VAN": 65, "YOZGAT": 66, "ZONGULDAK": 67, "AKSARAY": 68, "BAYBURT": 69, "KARAMAN": 70, "KIRIKKALE": 71, "BATMAN": 72,
    "ŞIRNAK": 73, "BARTIN": 74, "ARDAHAN": 75, "IĞDIR": 76, "YALOVA": 77, "KARABÜK": 78, "KİLİS": 79, "OSMANİYE": 80, "DÜZCE": 81
}

def plaka_ve_il_duzenle(df):
    """İl sütununu doldurur, plaka atar ve temizler."""
    if 'İl' in df.columns:
        # 1. Boş il isimlerini bir önceki dolu satırla doldur
        df['İl'] = df['İl'].ffill()
        
        # 2. İl isimlerini standardize et (Büyük harf ve temizlik)
        df['İl'] = df['İl'].astype(str).str.strip().str.upper()
        
        # 3. Plaka eşleştirme
        df['Plaka'] = df['İl'].map(plaka_kodlari)
        
        # 4. Plaka kodu bulunamayan veya 0/NaN olanları sil
        df = df.dropna(subset=['Plaka'])
        df = df[df['Plaka'] != 0]
        df['Plaka'] = df['Plaka'].astype(int)
        
        return df
    return df

# --- OPERASYON 1: TÜKETİCİ SAYISI ---
def islem_tuketici_sayisi():
    input_path = os.path.join("..","..", "processed_data", "steps", "04_2021_2024_subscriber_1.xlsx")
    output_path = os.path.join("..","..", "processed_data", "steps", "05_2021_2024_subscriber_2.xlsx")
    
    try:
        df = pd.read_excel(input_path)
        df.columns = df.columns.astype(str).str.strip()
        
        # Görseldeki 3. sütun adı hatasını (eksik harf vb.) gidermek için indeksle adlandır
        cols = list(df.columns)
        if len(cols) >= 3:
            cols[2] = "Tüketici Sayısı"
            df.columns = cols

        # İl ve Plaka İşlemleri
        df = plaka_ve_il_duzenle(df)

        # Tüketici Türü Temizliği
        if 'Tüketici Türü' in df.columns:
            df['Tüketici Türü'] = df['Tüketici Türü'].astype(str).str.strip()
            
            # Belirtilen değerlere sahip satırları sil
            yasakli_degerler = ["NaN", "nan","İl Toplam", "Genel Toplam", "Tüketici Türü"]
            df = df[~df['Tüketici Türü'].isin(yasakli_degerler)]
            
            # Sektör Gruplama
            mask_kamu = df['Tüketici Türü'].str.contains("Kamu|Ticarethane", na=False, case=False)
            df.loc[mask_kamu, 'Tüketici Türü'] = "Kamu"
            
            mask_tarim = df['Tüketici Türü'].str.startswith("Tarımsal", na=False)
            df.loc[mask_tarim, 'Tüketici Türü'] = "Tarım"

        # Kaynak_Dosya silme
        if 'Kaynak_Dosya' in df.columns:
            df.drop(columns=['Kaynak_Dosya'], inplace=True)

        # Sıralama
        df = df.sort_values(by=['İl', 'Yil', 'Ay', 'Tüketici Türü'])

        # Sütun Düzeni
        column_order = ['Plaka', 'İl', 'Yil', 'Ay', 'Tüketici Türü', 'Tüketici Sayısı']
        df = df[[c for c in column_order if c in df.columns]]

        df['Tüketici Sayısı'] = (
            df['Tüketici Sayısı']
            .astype(str)
            .str.split(',').str[0]  # Virgülden sonrasını çöpe atar
            .str.replace(".", "", regex=False)
            .replace('', '0') # Eğer hücre boşsa hata vermemesi için
            .astype(int)
        )

          
        df.to_excel(output_path, index=False)
        print(f"Tamamlandı: Tüketici Sayısı -> {output_path}")

    except Exception as e:
        print(f"Hata (Tüketici Sayısı): {e}")

# --- OPERASYON 2: TÜKETİM MİKTARI ---
def islem_tuketim_miktari():
    input_path = os.path.join("..", "..", "processed_data", "steps", "04_2021_2024_consumption_1.xlsx")
    output_path = os.path.join("..", "..", "processed_data", "steps", "05_2021_2024_consumption_2.xlsx")

    
    try:
        df = pd.read_excel(input_path)
        df.columns = df.columns.astype(str).str.strip()

        # Sadece "İl Toplam" satırlarını tut
        if 'Tüketici Türü' in df.columns:
            df = df[df['Tüketici Türü'].astype(str).str.strip() == "İl Toplam"].copy()

        # İl ve Plaka İşlemleri
        df = plaka_ve_il_duzenle(df)

        # Sütun Silme ve Adlandırma
        if 'Tüketici Türü' in df.columns: df.drop(columns=['Tüketici Türü'], inplace=True)
        if 'Kaynak_Dosya' in df.columns: df.drop(columns=['Kaynak_Dosya'], inplace=True)

        # 3. sütunu "Tüketim" yap (Plaka ve İl'den sonraki sütun)
        current_cols = list(df.columns)
        # Sütunlar: İl, Deger, Ay, Yil, Plaka (sıralama öncesi)
        if 'Deger' in df.columns:
            df.rename(columns={'Deger': 'Tüketim'}, inplace=True)

        # Sıralama ve Sütun Düzeni
        df = df.sort_values(by=['Plaka', 'Yil', 'Ay'])
        column_order = ['Plaka', 'İl', 'Yil', 'Ay', 'Tüketim']
        df = df[[c for c in column_order if c in df.columns]]


        # Tüketim verisi temizliği
        df["Tüketim"] = (
            df["Tüketim"]
            .astype(str)
            .str.split(',').str[0] # Virgülden sonrasını (kuruş/ondalık) siler
            .str.replace(".", "", regex=False)
            .replace('', '0')
            .astype(int)
        )


        
        df.to_excel(output_path, index=False)
        print(f"Tamamlandı: Tüketim Miktarı -> {output_path}")

    except Exception as e:
        print(f"Hata (Tüketim Miktarı): {e}")

if __name__ == "__main__":
    islem_tuketici_sayisi()
    islem_tuketim_miktari()

Tamamlandı: Tüketici Sayısı -> ..\..\processed_data\steps\05_2021_2024_subscriber_2.xlsx
Tamamlandı: Tüketim Miktarı -> ..\..\processed_data\steps\05_2021_2024_consumption_2.xlsx
